In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "DTD_Replication_Study" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_23331694_survey_sql <- paste("
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `cb_criteria` cr                               
                WHERE
                    concept_id IN (40192389, 1585855, 1586134)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT ca.descendant_id 
                        FROM
                            `cb_criteria_ancestor` ca 
                        JOIN
                            (SELECT
                                DISTINCT c.concept_id       
                            FROM
                                `cb_criteria` c       
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id             
                                FROM
                                    `cb_criteria` cr             
                                WHERE
                                    concept_id IN (21604686, 21604788, 21604729)             
                                    AND full_text LIKE '%_rank1]%'       ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) b 
                                ON (ca.ancestor_id = b.concept_id)) 
                            AND is_standard = 1)) criteria ) ))", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_23331694_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "survey_23331694",
  "survey_23331694_*.csv")
message(str_glue('The data will be written to {survey_23331694_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_23331694_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_23331694_path,
  destination_format = "CSV")

In [ ]:
library(tidyverse)
library(bigrquery)

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_23331694_path}` to copy these files
#       to the Jupyter disk.

survey_23331694_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260618", 
  "survey_23331694",
  "survey_23331694_*.csv")
message(str_glue('The data will be written to {survey_23331694_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character(), survey_version_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_23331694_survey_df <- read_bq_export_from_workspace_bucket(survey_23331694_path)

dim(dataset_23331694_survey_df)

head(dataset_23331694_survey_df, 5)

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "DTD_Replication_Study" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_23331694_person_sql <- paste("
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT ca.descendant_id 
                    FROM
                        `cb_criteria_ancestor` ca 
                    JOIN
                        (SELECT
                            DISTINCT c.concept_id       
                        FROM
                            `cb_criteria` c       
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id             
                            FROM
                                `cb_criteria` cr             
                            WHERE
                                concept_id IN (21604686, 21604788, 21604729)             
                                AND full_text LIKE '%_rank1]%'       ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) b 
                            ON (ca.ancestor_id = b.concept_id)) 
                        AND is_standard = 1)) criteria ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_23331694_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_23331694",
  "person_23331694_*.csv")
message(str_glue('The data will be written to {person_23331694_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_23331694_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_23331694_path,
  destination_format = "CSV")

In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_23331694_path}` to copy these files
#       to the Jupyter disk.

person_23331694_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260618", # Comment out this line if you want the export to always overwrite.
  "person_23331694",
  "person_23331694_*.csv")
message(str_glue('The data will be written to {person_23331694_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(gender = col_character(), race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_23331694_person_df <- read_bq_export_from_workspace_bucket(person_23331694_path)

dim(dataset_23331694_person_df)

head(dataset_23331694_person_df, 5)

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "DTD_Replication_Study" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_23331694_measurement_sql <- paste("
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `measurement` measurement 
        WHERE
            (
                measurement_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (903124)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604686, 21604788, 21604729)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) measurement 
        LEFT JOIN
            `concept` m_standard_concept 
                ON measurement.measurement_concept_id = m_standard_concept.concept_id 
        LEFT JOIN
            `concept` m_type 
                ON measurement.measurement_type_concept_id = m_type.concept_id 
        LEFT JOIN
            `concept` m_operator 
                ON measurement.operator_concept_id = m_operator.concept_id 
        LEFT JOIN
            `concept` m_value 
                ON measurement.value_as_concept_id = m_value.concept_id 
        LEFT JOIN
            `concept` m_unit 
                ON measurement.unit_concept_id = m_unit.concept_id 
        LEFT JOIn
            `visit_occurrence` v 
                ON measurement.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` m_visit 
                ON v.visit_concept_id = m_visit.concept_id 
        LEFT JOIN
            `concept` m_source_concept 
                ON measurement.measurement_source_concept_id = m_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurement_23331694_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "measurement_23331694",
  "measurement_23331694_*.csv")
message(str_glue('The data will be written to {measurement_23331694_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_23331694_measurement_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  measurement_23331694_path,
  destination_format = "CSV")

In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurement_23331694_path}` to copy these files
#       to the Jupyter disk.

measurement_23331694_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260618",  # Comment out this line if you want the export to always overwrite.
  "measurement_23331694",
  "measurement_23331694_*.csv")
message(str_glue('The data will be written to {measurement_23331694_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), measurement_type_concept_name = col_character(), operator_concept_name = col_character(), value_as_concept_name = col_character(), unit_concept_name = col_character(), visit_occurrence_concept_name = col_character(), measurement_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), unit_source_value = col_character(), value_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_23331694_measurement_df <- read_bq_export_from_workspace_bucket(measurement_23331694_path)

dim(dataset_23331694_measurement_df)

head(dataset_23331694_measurement_df, 5)

In [ ]:
# This query represents dataset "Migraine_Insomnia_w_AD" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_17661305_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (318736, 436962)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604686, 21604788, 21604729)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_17661305_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_17661305",
  "condition_17661305_*.csv")
message(str_glue('The data will be written to {condition_17661305_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_17661305_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_17661305_path,
  destination_format = "CSV")

In [ ]:
condition_17661305_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260619",  # Comment out this line if you want the export to always overwrite.
  "condition_17661305",
  "condition_17661305_*.csv")
message(str_glue('The data will be written to {condition_17661305_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_17661305_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_17661305_condition_df <- read_bq_export_from_workspace_bucket(condition_17661305_path)

dim(dataset_17661305_condition_df)

head(dataset_17661305_condition_df, 5)

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "PsychoticDisorders_w_AD" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_27607955_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (436073)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604686, 21604788, 21604729)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_27607955_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_27607955",
  "condition_27607955_*.csv")
message(str_glue('The data will be written to {condition_27607955_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_27607955_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_27607955_path,
  destination_format = "CSV")

In [ ]:
condition_27607955_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260619",  # Comment out this line if you want the export to always overwrite.
  "condition_27607955",
  "condition_27607955_*.csv")
message(str_glue('The data will be written to {condition_27607955_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_27607955_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_27607955_condition_df <- read_bq_export_from_workspace_bucket(condition_27607955_path)

dim(dataset_27607955_condition_df)

head(dataset_27607955_condition_df, 5)

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "BIP_w_AD" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_04192082_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (436665)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604686, 21604788, 21604729)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_04192082_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_04192082",
  "condition_04192082_*.csv")
message(str_glue('The data will be written to {condition_04192082_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_04192082_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_04192082_path,
  destination_format = "CSV")

In [ ]:
condition_04192082_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260619",  # Comment out this line if you want the export to always overwrite.
  "condition_04192082",
  "condition_04192082_*.csv")
message(str_glue('The data will be written to {condition_04192082_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_04192082_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_04192082_condition_df <- read_bq_export_from_workspace_bucket(condition_04192082_path)

dim(dataset_04192082_condition_df)

head(dataset_04192082_condition_df, 5)

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "MDD_w_AD" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_23855060_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (4152280)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604686, 21604788, 21604729)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_23855060_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_23855060",
  "condition_23855060_*.csv")
message(str_glue('The data will be written to {condition_23855060_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_23855060_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_23855060_path,
  destination_format = "CSV")

In [ ]:
condition_23855060_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260619",  # Comment out this line if you want the export to always overwrite.
  "condition_23855060",
  "condition_23855060_*.csv")
message(str_glue('The data will be written to {condition_23855060_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_23855060_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_23855060_condition_df <- read_bq_export_from_workspace_bucket(condition_23855060_path)

dim(dataset_23855060_condition_df)

head(dataset_23855060_condition_df, 5)

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "Anxiety_w_AD" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_88555975_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (441542)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604686, 21604788, 21604729)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_88555975_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_88555975",
  "condition_88555975_*.csv")
message(str_glue('The data will be written to {condition_88555975_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_88555975_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_88555975_path,
  destination_format = "CSV")

In [ ]:
condition_88555975_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260619",  # Comment out this line if you want the export to always overwrite.
  "condition_88555975",
  "condition_88555975_*.csv")
message(str_glue('The data will be written to {condition_88555975_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_88555975_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_88555975_condition_df <- read_bq_export_from_workspace_bucket(condition_88555975_path)

dim(dataset_88555975_condition_df)

head(dataset_88555975_condition_df, 5)

In [ ]:
#--Smoking
survey <- dataset_23331694_survey_df

library(dplyr)

smoking <- survey %>%
  group_by(person_id) %>%
  summarise(
    Smoking_Frequency = case_when(
      any(answer_concept_id == 1585861) ~ 4L,  # Every Day
      any(answer_concept_id == 1585862) ~ 3L,  # Some Days
      any(answer_concept_id == 1585863) ~ 2L,  # Not At All (former)
      any(answer_concept_id == 1585859) ~ 1L,  # No  -> lifetime abstainer
      TRUE                              ~ NA_integer_
    ),
    .groups = "drop"
  ) %>%
  mutate(Smoking_Frequency = factor(
    Smoking_Frequency, levels = 1:4,
    labels = c("Lifetime Abstainer", "Not At All (former)", "Some Days", "Every Day")
  ))

# Alcohol
alcohol <- survey %>%
  group_by(person_id) %>%
  summarise(
    Alcohol_Frequency = case_when(
      any(answer_concept_id == 1586206) ~ 6L,  # >=4 Per Week
      any(answer_concept_id == 1586205) ~ 5L,  # 2-3 Per Week
      any(answer_concept_id == 1586204) ~ 4L,  # 2-4 Per Month
      any(answer_concept_id == 1586203) ~ 3L,  # Monthly Or Less
      any(answer_concept_id == 1586202) ~ 2L,  # Never (former / not past year)
      any(answer_concept_id == 1586200) ~ 1L,  # No -> lifetime abstainer
      TRUE                              ~ NA_integer_
    ),
    .groups = "drop"
  ) %>%
  mutate(Alcohol_Frequency = factor(
    Alcohol_Frequency, levels = 1:6,
    labels = c("Lifetime Abstainer", "Never (former)", "Monthly or Less",
               "2-4 Month", "2-3 Week", "4plus Week")
  ))

In [ ]:
income_map <- c(
  "1585376" = 1L,  # less 10k   -> <=25k
  "1585377" = 1L,  # 10k-25k    -> <=25k
  "1585378" = 2L,  # 25k-35k
  "1585379" = 3L,  # 35k-50k
  "1585380" = 4L,  # 50k-75k
  "1585381" = 5L,  # 75k-100k
  "1585382" = 6L,  # 100k-150k
  "1585383" = 7L,  # 150k-200k
  "1585384" = 8L   # more 200k
  # 903079 (Prefer Not To Answer) and 903096 (Skip) -> NA by omission
)

income <- survey %>%
  filter(answer_concept_id %in% names(income_map)) %>%
  distinct(person_id, answer_concept_id) %>%
  mutate(Income = income_map[as.character(answer_concept_id)]) %>%
  select(person_id, Income) %>%
  mutate(Income = factor(
    Income, levels = 1:8,
    labels = c("<=25k", "25-35k", "35-50k", "50-75k",
               "75-100k", "100-150k", "150-200k", ">200k")
  ))

educ_map <- c(
  "1585941" = 1L,  # Never Attended      -> Less than HS
  "1585942" = 1L,  # One Through Four    -> Less than HS
  "1585943" = 1L,  # Five Through Eight  -> Less than HS
  "1585944" = 1L,  # Nine Through Eleven -> Less than HS
  "1585945" = 2L,  # Twelve Or GED       -> HS / GED
  "1585946" = 3L,  # College One to Three-> Undergraduate
  "1585947" = 3L,  # College Graduate    -> Undergraduate
  "1585948" = 4L   # Advanced Degree     -> Postgraduate
  # 903079 (Prefer Not To Answer) and 903096 (Skip) -> NA by omission
)

education <- survey %>%
  filter(answer_concept_id %in% names(educ_map)) %>%
  distinct(person_id, answer_concept_id) %>%
  mutate(Education_level = educ_map[as.character(answer_concept_id)]) %>%
  select(person_id, Education_level) %>%
  mutate(Education_level = factor(
    Education_level, levels = 1:4,
    labels = c("Less than a high school degree", "Twelve or GED",
               "College Undergraduate", "College Postgraduate")
  ))

stress_map <- c(
  "40192465" = 1L,  # Never
  "40192430" = 2L,  # Almost Never
  "40192429" = 3L,  # Sometimes
  "40192477" = 4L,  # Fairly Often
  "40192424" = 5L   # Very Often
  # 903096 (Skip) -> NA by omission
)

stress <- survey %>%
  filter(answer_concept_id %in% names(stress_map)) %>%
  group_by(person_id) %>%
  slice_max(survey_datetime, n = 1, with_ties = FALSE) %>%   # latest wave
  ungroup() %>%
  mutate(Stress = stress_map[as.character(answer_concept_id)]) %>%
  select(person_id, Stress) %>%
  mutate(Stress = factor(Stress, levels = 1:5, ordered = TRUE,
    labels = c("Never","Almost Never","Sometimes","Fairly Often","Very Often")))

In [ ]:
pheno_all <- smoking %>%
  full_join(education, by = "person_id") %>%
  full_join(alcohol, by = "person_id") %>%
  full_join(income, by = "person_id") %>%
  full_join(stress, by = "person_id")

In [ ]:
#-- sex at birth and DOB
person <- dataset_23331694_person_df
person <- person %>%
  select(person_id, date_of_birth, sex_at_birth) %>%
  mutate(
    sex_at_birth = case_when(
      !sex_at_birth %in% c("Female", "Male") ~ NA,
      TRUE ~ sex_at_birth
    )
  )

In [ ]:
# BMI (median value)
BMI <- dataset_23331694_measurement_df %>%
    group_by(person_id) %>%
    summarise(BMI = median(value_as_number, na.rm = TRUE)) %>%
    ungroup()

In [ ]:
#-- combine rest
pheno_all <- pheno_all %>%
  full_join(person, by = "person_id") %>%
  full_join(BMI, by = "person_id")

In [ ]:
#-- Conditions
psy <- dataset_27607955_condition_df %>% mutate(Condition = "Psychotic")
bip <- dataset_04192082_condition_df %>% mutate(Condition = "BIP")
mdd <- dataset_23855060_condition_df %>% mutate(Condition = "MDD")
anx <- dataset_88555975_condition_df %>% mutate(Condition = "ANX")
mig_ins <- dataset_17661305_condition_df
mig <- mig_ins %>%
  filter(standard_concept_code != "193462001")
ins <- mig_ins %>%
  filter(standard_concept_code == "193462001")

ins_cnt <- ins %>%
  group_by(person_id) %>%
  summarise(
    ins_count = n()
  )

mig_cnt <- mig %>%
  group_by(person_id) %>%
  summarise(
    mig_count = n()
  )

mdd_cnt <- mdd %>%
  group_by(person_id) %>%
  summarise(
    mdd_count = n()
  )

anx_cnt <- anx %>%
  group_by(person_id) %>%
  summarise(
    anx_count = n()
  )

bip_cnt <- bip %>%
    group_by(person_id) %>%
    summarise(
        bip_count = n()
    )

psy_cnt <- psy %>%
    group_by(person_id) %>%
    summarise(
        psy_cnt = n()
    )


In [ ]:
#-- Map conditions back to person_pheno df
pheno_all <- pheno_all %>%
  mutate(
    Depression = ifelse(person_id %in% mdd_cnt$person_id, 1, 0),
    Anxiety = ifelse(person_id %in% anx_cnt$person_id, 1, 0),
    BIP = ifelse(person_id %in% bip_cnt$person_id, 1, 0),
    Psychotic = ifelse(person_id %in% psy_cnt$person_id, 1, 0),
    Insomnia = ifelse(person_id %in% ins_cnt$person_id, 1, 0),
    Migraine = ifelse(person_id %in% mig_cnt$person_id, 1, 0)
  )
head(pheno_all)

In [ ]:
# read in the CIDI-sf

library(bigrquery)
library(stringr)
library(tidyverse)

download_data <- function(query) {
  tb <- bq_project_query(Sys.getenv('GOOGLE_PROJECT'), query)
  bq_table_download(tb, bigint = "integer64")
}

MHWB_DATASET <- 'fc-aou-cdr-prod-ct.C_V8_R2_offcycle_mhwb'   # hardcoded, Controlled Tier

In [ ]:
# the query helper
mhwb_sql <- function(question_concept_id) {
  ids <- paste(question_concept_id, collapse = ",")
  query <- str_glue("
    SELECT DISTINCT obs.person_id,
                    observation_source_concept_id AS question_id,
                    con1.concept_name AS question,
                    value_source_concept_id AS answer_id,
                    con2.concept_name AS answer
    FROM `{MHWB_DATASET}.observation` obs
    JOIN `{MHWB_DATASET}.concept` con2 ON obs.value_source_concept_id = con2.concept_id
    JOIN `{MHWB_DATASET}.concept` con1 ON obs.observation_source_concept_id = con1.concept_id
    WHERE observation_source_concept_id IN ({ids})
  ")
  download_data(query)
}

In [ ]:
cidi_ids   <- c(1704045, 1703998, 1704047, 1704036, 1704012,
                1704027, 1704025, 1704136, 1704022, 1704002)
cidi_label <- c("A1","A2","A3","A4","A6","A7","A8","A9","B","PPD")

cidi_df <- NULL
for (i in seq_along(cidi_ids)) {
  d <- mhwb_sql(cidi_ids[i]) |>
    select(person_id, answer) |>
    distinct(person_id, .keep_all = TRUE)
  colnames(d) <- c("person_id", cidi_label[i])
  cidi_df <- if (is.null(cidi_df)) d
             else full_join(cidi_df, d, by = "person_id", relationship = "one-to-one")
  cat(sprintf("Pulled %s (%d rows)\n", cidi_label[i], nrow(d)))
}
dim(cidi_df)

Map DSM MDD Criteria

Create an indicator for persons who can be scored

We require at least one of either A1 or A2 be non-missing and answered either Yes or No.

i.e., We'd consider MDD "missing" for persons who skipped answering both A1 and A2 because we cannot score them.

In [ ]:
cidi_df_clean <- cidi_df |> 
  mutate(ToScore = ifelse(
    ( is.na(A1) & is.na(A2) ) |
      (is.na(A1) & A2 %in% "PMI: Skip") |
      (is.na(A2) & A1 %in% "PMI: Skip") |
      (A1 %in% "PMI: Skip" & A2 %in% "PMI: Skip"),
    0, 1
  ))

cidi_df_clean |> 
  count(ToScore,A1,A2)
cidi_df_clean |> 
  count(ToScore)

In [ ]:
cidi_df_clean <- cidi_df_clean |> 
  mutate(
    A1score = ifelse(A1 %in% "Yes", 1, 0),
    A2score = ifelse(A2 %in% "Yes", 1, 0),
    A3score = ifelse(A3 %in% c("Gained weight", "Lost weight", "Both gained and lost some weight during the episode"), 1, 0),
    A4score = ifelse(A4 %in% "Yes", 1, 0),
    A6score = ifelse(A6 %in% "Yes", 1, 0),
    A7score = ifelse(A7 %in% "Yes", 1, 0),
    A8score = ifelse(A8 %in% "Yes", 1, 0),
    A9score = ifelse(A9 %in% "Yes", 1, 0),
  )

table(cidi_df_clean$A1score, cidi_df_clean$A1, useNA = "ifany")
table(cidi_df_clean$A2score, cidi_df_clean$A2, useNA = "ifany")
table(cidi_df_clean$A3, cidi_df_clean$A3score, useNA = "ifany")
table(cidi_df_clean$A4score, cidi_df_clean$A4, useNA = "ifany")
table(cidi_df_clean$A6score, cidi_df_clean$A6, useNA = "ifany")
table(cidi_df_clean$A7score, cidi_df_clean$A7, useNA = "ifany")
table(cidi_df_clean$A8score, cidi_df_clean$A8, useNA = "ifany")
table(cidi_df_clean$A9score, cidi_df_clean$A9, useNA = "ifany")

In [ ]:
cidi_df_clean <- cidi_df_clean |> 
  mutate(cardinal = ifelse((A1 %in% "Yes" | A2 %in% "Yes"), 
                           1, 0),
         cardinal = ifelse(ToScore == 0, 
                           # These individuals having missing info, not a zero score for cardinal symptoms
                           NA, cardinal))

## Check
cidi_df_clean |> 
  count(cardinal, A1, A2)

cidi_df_clean |> 
  count(cardinal)

In [ ]:
cidi_df_clean <- cidi_df_clean |> 
  mutate(TotalScore = A1score + A2score + A3score + A4score + A6score + A7score + A8score + A9score,
         TotalScore = ifelse(ToScore == 0, NA, TotalScore))

table(cidi_df_clean$TotalScore, useNA="ifany")
table(cidi_df_clean$TotalScore[cidi_df_clean$cardinal == 1])

In [ ]:
cidi_df_clean <- cidi_df_clean |>
  mutate(A_criterion= case_when(
    ToScore == 0            ~ NA_real_,          # unscoreable stays missing
    cardinal == 1 & TotalScore >= 5 ~ 1,         # cardinal symptom + ≥5 total
    TRUE                    ~ 0                   # scoreable but doesn't meet criteria
  ))

table(cidi_df_clean$A_criterion)

In [ ]:
impair_yes <- c("A lot", "Somewhat")   # see decision below — set and document this

cidi_df_clean <- cidi_df_clean |>
  mutate(
    B_impair = case_when(
      B %in% impair_yes              ~ 1,
      B %in% c("A little", "Not at all") ~ 0,
      TRUE                           ~ NA_real_   # PMI: Skip and NA
    ),
    MDD = case_when(
      ToScore == 0                          ~ NA_real_,   # unscoreable on stems
      A_criterion == 0                      ~ 0,          # failed symptoms → non-case, B irrelevant
      A_criterion == 1 & B_impair == 1      ~ 1,          # symptoms + impairment → case
      A_criterion == 1 & B_impair == 0      ~ 0,          # symptoms but no impairment → non-case
      A_criterion == 1 & is.na(B_impair)    ~ NA_real_    # passed A but B truly missing → can't finalise
    )
  )

cidi_df_clean |> count(MDD)
cidi_df_clean |> count(A_criterion, B_impair)   # diagnostic cross-tab

In [ ]:
### Define MDD Inclusion variable based on A + B criteria

### Both A and B met = Cases
### A met, but B not met = Excluded (Sub-threshold)
### B met, A not met = Excluded (Sub-threshold)
### A not met, B not responded = Controls.
### A met, B not responded = Excluded (Insufficient Info)
### Neither A nor B met = Controls.

cidi_df_clean <- cidi_df_clean |>
  mutate(MDD_Incl = case_when(
    ToScore == 0                        ~ NA_real_,
    A_criterion == 1 & B_impair == 1    ~ 1,
    A_criterion == 0 & B_impair == 0    ~ 0,
    A_criterion == 0 & is.na(B_impair)  ~ 0,    # screened negative on symptoms → control
    A_criterion == 1 & B_impair == 0    ~ NA_real_,
    A_criterion == 0 & B_impair == 1    ~ NA_real_,
    is.na(B_impair)                     ~ NA_real_,
    TRUE                                ~ NA_real_
  ),
    MDD_symptom_count = ifelse(ToScore == 1, TotalScore, NA_real_),)

table(cidi_df_clean$MDD_Incl, useNA = "ifany")
cidi_df_select <- cidi_df_clean %>%
    select(person_id, MDD_Incl, MDD_symptom_count)

In [ ]:
# join with phenotypes
cidi_df_select <- cidi_df_select |> mutate(person_id = as.character(person_id))
pheno_all       <- pheno_all       |> mutate(person_id = as.character(person_id))
pheno_new <- pheno_all %>%
    left_join(cidi_df_select, by = "person_id")
dim(pheno_new); head(pheno_new)

In [ ]:
# Save locally first
write.csv(pheno_new, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv", row.names = FALSE)

# Copy to workspace bucket
system(paste(
  "gsutil -u", Sys.getenv("GOOGLE_PROJECT"), "cp /home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv",
  file.path(Sys.getenv("WORKSPACE_BUCKET"), "results/Phenotypes.csv")
))